# NLP [Natural Language Processing]:


- **Steps involved in NLP with the help of nltk library as follow:**

 
    1. Noise Removal [URLs, Emails, tags, special chars, etc...].
    2. Remove punctuations & extra whitespaces.
    3. Stop words exclusion.
    4. Tokenization.
    5. Stemming & Lemmatization.
    6. Part-Of-Speech tagging [Optional - feature extraction]
    7. Text Vectorization [BoW, Tf-IDF, Word Embeddings (Word2Vec, BERT,...), etc...]

---

Basic Implementation guide:

In [2]:
import os, warnings
from collections import Counter
from typing import List, Tuple
import numpy as np, pandas as pd
import seaborn as sns, matplotlib.pyplot as plt
import random

import nltk
import re

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer

warnings.filterwarnings('ignore')
%matplotlib inline

In [3]:
# nltk.download('punkt')
# nltk.download('stopwords')
# nltk.download('wordnet')
# nltk.download('averaged_perceptron_tagger')

Practice Dataset Loading:

- Kaggle: [Email Spam Detection Dataset 🔗](https://www.kaggle.com/datasets/venky73/spam-mails-dataset/data)

In [4]:
data = pd.read_csv(os.path.join('dataset', 'spam_ham_dataset.csv'))
data.head(3)

,Unnamed: 0,label,text,label_num
0,605,ham,Subject: enron methanol ; meter # : 988291\r\n...,0
1,2349,ham,"Subject: hpl nom for january 9 , 2001\r\n( see...",0
2,3624,ham,"Subject: neon retreat\r\nho ho ho , we ' re ar...",0


In [5]:
i = np.random.randint(0, data.shape[0]-1)
text = data['text'][i]
print(text)

Subject: unify close schedule
the following is the close schedule for this coming month ( year - end . ) please
keep in the mind the following key times . . . .
unify to sitara bridge back 1 : 45 p . m . thursday , dec 30 th ( all errors must be
clear by this time )
mass draft at 6 p . m . thursday evening , dec 30 th .
accrual process begins friday morning , dec 31 st at 6 : 30 a . m . ( if your group
impacts the accrual , please ensure that the necessary people are available
for support if needed , as this is an enron holiday . )
please feel free to contact me should you have any questions .
thank you , melissa x 35615


---

In [6]:
# Noise Removal Regular Expressions:
URL_NOISE_RE = r"http?:\S+|https?:\S+|www\S+"
EMAIL_NOISE_RE = r"\S+@\S+"
HTML_TAGS_NOISE_RE = r"<.*?>"
DIG_SC_RE = r"[^a-z\s]"
WHITESPACE_RE = r"\s+"

# Stop words: 
STOP_WORDS = set(stopwords.words('english'))

# Stemmers & Lemmatizers:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

In [7]:
def preprocess_text(text) -> Tuple[str, List[str]]:
    
    
    # 1. Noise & whitespace removal...
    text = text.lower()
    text = re.sub(URL_NOISE_RE, '', text)
    text = re.sub(EMAIL_NOISE_RE, '', text)
    text = re.sub(HTML_TAGS_NOISE_RE, '', text)
    text = re.sub(r'subject', '', text)
    text = re.sub(DIG_SC_RE, '', text)
    text = re.sub(WHITESPACE_RE, ' ', text).strip()
    
    # 2. Tokenization...
    tokens = word_tokenize(text)
     
    # 3. Stop Words Exclusion
    tokens = [word for word in tokens if word not in STOP_WORDS]
    
    # 4. Stemming.
    stemmed = [stemmer.stem(word) for word in tokens]
    
    # 5. Lemmatization.
    lemmatized = [lemmatizer.lemmatize(word) for word in stemmed]

    return " ".join(lemmatized), lemmatized

In [8]:
preprocessed_text, tokens = preprocess_text(text)
print(preprocessed_text)
print(f"Tokens Length: {len(tokens)}")

unifi close schedul follow close schedul come month year end plea keep mind follow key time unifi sitara bridg back p thursday dec th error must clear time mass draft p thursday even dec th accrual process begin friday morn dec st group impact accrual plea ensur necessari peopl avail support need enron holiday plea feel free contact question thank melissa x
Tokens Length: 62


Optional:

- POS Tagging.

In [9]:
def POS_Tagging(tokens: List[str]) -> List[Tuple[str]]:
    return nltk.pos_tag(tokens)

In [10]:
tagged_tokens = POS_Tagging(tokens)
tagged_tokens[:10]

[('unifi', 'JJ'),
 ('close', 'RB'),
 ('schedul', 'JJ'),
 ('follow', 'VBP'),
 ('close', 'JJ'),
 ('schedul', 'NNS'),
 ('come', 'VBP'),
 ('month', 'NN'),
 ('year', 'NN'),
 ('end', 'NN')]

In [11]:
# Tags Counter:

def pos_features(tagged_tokens) -> int:
    pos_tags = [pos for _, pos in tagged_tokens]
    return Counter(pos_tags)

In [12]:
pos_features(tagged_tokens)

Counter({'NN': 34,
         'JJ': 12,
         'RB': 4,
         'VBP': 3,
         'VB': 3,
         'VBN': 2,
         'NNS': 1,
         'IN': 1,
         'MD': 1,
         'VBD': 1})

---
Convert the tokens [words] into the vectors and later can concat the POS features into the vectors...

In [39]:
from sklearn.preprocessing import StandardScaler

In [13]:
data[['preprocessed_txt', 'tokens']] = data['text'].apply(lambda x:pd.Series(preprocess_text(x)))

In [14]:
df = data[['preprocessed_txt', 'tokens', 'label']]

In [15]:
df['pos_tags'] = df['tokens'].apply(POS_Tagging)
df['pos_freq'] = df['pos_tags'].apply(pos_features)

In [16]:
df.head(3)

,preprocessed_txt,tokens,label,pos_tags,pos_freq
0,enron methanol meter follow note gave monday p...,"[enron, methanol, meter, follow, note, gave, m...",ham,"[(enron, NN), (methanol, NN), (meter, NN), (fo...","{'NN': 17, 'JJ': 7, 'VBD': 1, 'NNS': 2, 'CD': ..."
1,hpl nom januari see attach file hplnol xl hpln...,"[hpl, nom, januari, see, attach, file, hplnol,...",ham,"[(hpl, NN), (nom, NN), (januari, NN), (see, VB...","{'NN': 6, 'VBP': 1, 'RB': 1, 'JJ': 1, 'NNP': 1}"
2,neon retreat ho ho ho around wonder time year ...,"[neon, retreat, ho, ho, ho, around, wonder, ti...",ham,"[(neon, RB), (retreat, NN), (ho, JJ), (ho, NN)...","{'RB': 10, 'NN': 98, 'JJ': 45, 'IN': 7, 'JJR':..."


In [44]:
pos_df = pd.DataFrame(df['pos_freq'].tolist()).fillna(0)
pos_df.head(5)

# Scaling POS:

scaler = StandardScaler()
pos_ = scaler.fit_transform(pos_df)
pos_df = pd.DataFrame(pos_, columns=scaler.get_feature_names_out())
pos_df.head(5)

,NN,JJ,VBD,NNS,CD,VB,VBP,RB,NNP,IN,...,WDT,SYM,NNPS,PRP$,WP$,POS,EX,PDT,UH,''
0,-0.447180,-0.329835,-0.166454,0.335185,0.333264,-0.370797,-0.435047,-0.538616,-0.378670,-0.484231,...,-0.025888,-0.052103,-0.01967,-0.063857,-0.059499,-0.01967,-0.024093,-0.01967,-0.037763,-0.013908
1,-0.557230,-0.561053,-0.447555,-0.539306,-0.344937,-0.627304,-0.435047,-0.270449,0.257284,-0.484231,...,-0.025888,-0.052103,-0.01967,-0.063857,-0.059499,-0.01967,-0.024093,-0.01967,-0.037763,-0.013908
2,0.363190,1.134545,0.114648,0.772431,1.011465,2.450783,1.373678,2.143057,-0.378670,1.813279,...,-0.025888,-0.052103,-0.01967,-0.063857,-0.059499,-0.01967,-0.024093,-0.01967,-0.037763,-0.013908
3,-0.347134,-0.137154,-0.166454,-0.539306,-0.344937,-0.627304,-0.435047,-0.538616,-0.378670,-0.484231,...,-0.025888,-0.052103,-0.01967,-0.063857,-0.059499,-0.01967,-0.024093,-0.01967,-0.037763,-0.013908
4,-0.457184,-0.368372,-0.447555,-0.102061,-0.344937,-0.370797,-0.328652,-0.538616,-0.378670,0.172201,...,-0.025888,-0.052103,-0.01967,-0.063857,-0.059499,-0.01967,-0.024093,-0.01967,-0.037763,-0.013908


In [45]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [46]:
vectorizer = TfidfVectorizer(ngram_range=(1,2), max_features=964)
X_tfidf = vectorizer.fit_transform(df['preprocessed_txt'])
X_tfidf.shape

# Converting to df...
tfidf_df = pd.DataFrame(X_tfidf.toarray(), columns=vectorizer.get_feature_names_out())
tfidf_df.head(2)

,abl,accept,access,account,act,action,activ,acton,actual,actual flow,...,xl hplno,xp,yahoo,yahoo com,year,yet,young,young hou,zero,zone
0,0.0,0.0,0.0,0.0,0.0,0.0,0.208212,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.239388,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0


---

In [47]:
final_features = pd.concat([tfidf_df.reset_index(drop=True), pos_df.reset_index(drop=True)], axis=1)
final_features

,abl,accept,access,account,act,action,activ,acton,actual,actual flow,...,WDT,SYM,NNPS,PRP$,WP$,POS,EX,PDT,UH,''
0,0.0,0.0,0.0,0.00000,0.0,0.0,0.208212,0.0,0.0,0.0,...,-0.025888,-0.052103,-0.01967,-0.063857,-0.059499,-0.01967,-0.024093,-0.01967,-0.037763,-0.013908
1,0.0,0.0,0.0,0.00000,0.0,0.0,0.000000,0.0,0.0,0.0,...,-0.025888,-0.052103,-0.01967,-0.063857,-0.059499,-0.01967,-0.024093,-0.01967,-0.037763,-0.013908
2,0.0,0.0,0.0,0.00000,0.0,0.0,0.000000,0.0,0.0,0.0,...,-0.025888,-0.052103,-0.01967,-0.063857,-0.059499,-0.01967,-0.024093,-0.01967,-0.037763,-0.013908
3,0.0,0.0,0.0,0.00000,0.0,0.0,0.000000,0.0,0.0,0.0,...,-0.025888,-0.052103,-0.01967,-0.063857,-0.059499,-0.01967,-0.024093,-0.01967,-0.037763,-0.013908
4,0.0,0.0,0.0,0.00000,0.0,0.0,0.000000,0.0,0.0,0.0,...,-0.025888,-0.052103,-0.01967,-0.063857,-0.059499,-0.01967,-0.024093,-0.01967,-0.037763,-0.013908
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5166,0.0,0.0,0.0,0.00000,0.0,0.0,0.000000,0.0,0.0,0.0,...,-0.025888,-0.052103,-0.01967,-0.063857,-0.059499,-0.01967,-0.024093,-0.01967,-0.037763,71.902712
5167,0.0,0.0,0.0,0.00000,0.0,0.0,0.000000,0.0,0.0,0.0,...,-0.025888,-0.052103,-0.01967,-0.063857,-0.059499,-0.01967,-0.024093,-0.01967,-0.037763,-0.013908
5168,0.0,0.0,0.0,0.00000,0.0,0.0,0.000000,0.0,0.0,0.0,...,-0.025888,-0.052103,-0.01967,-0.063857,-0.059499,-0.01967,-0.024093,-0.01967,-0.037763,-0.013908
5169,0.0,0.0,0.0,0.00000,0.0,0.0,0.265612,0.0,0.0,0.0,...,-0.025888,-0.052103,-0.01967,-0.063857,-0.059499,-0.01967,-0.024093,-0.01967,-0.037763,-0.013908


In [48]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [49]:
label_encoder = LabelEncoder()

In [50]:
X = final_features
y = label_encoder.fit_transform(df['label'])

In [51]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.3, random_state=45)

In [52]:
model = LogisticRegression()
model.fit(X_train, y_train)

LogisticRegression()

In [53]:
y_pred = model.predict(X_test)

In [54]:
print(f"""Accuracy: {accuracy_score(y_test, y_pred)}
{classification_report(y_test, y_pred)}""")

Accuracy: 0.9600515463917526
              precision    recall  f1-score   support

           0       0.97      0.97      0.97      1107
           1       0.93      0.93      0.93       445

    accuracy                           0.96      1552
   macro avg       0.95      0.95      0.95      1552
weighted avg       0.96      0.96      0.96      1552



---